# 01 — Data Loading & Merging

## Mục tiêu
- Load dữ liệu từ nhiều file CSV (theo quận)
- Thêm cột `source` để truy vết nguồn dữ liệu
- Chuẩn hóa tên cột
- Gộp (merge) thành 1 dataset duy nhất
- Lưu file kết quả vào `data/processed/`

In [1]:
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings("ignore")

In [2]:

DATA_PATH = "../data/raw/"
OUTPUT_PATH = "../data/processed/"

In [3]:
data_binh_thanh = pd.read_csv(DATA_PATH + "quan-binh-thanh.csv")
data_go_vap = pd.read_csv(DATA_PATH + "quan-go-vap.csv")
data_phu_nhuan = pd.read_csv(DATA_PATH + "quan-phu-nhuan.csv")

In [4]:
print(data_binh_thanh.shape)
print(data_go_vap.shape)
print(data_phu_nhuan.shape)

(2665, 23)
(4374, 23)
(1234, 23)


### Nhận xét

- **Bình Thạnh**: 2,665 tin đăng
- **Gò Vấp**: 4,374 tin đăng (nhiều nhất)
- **Phú Nhuận**: 1,234 tin đăng (ít nhất)

Mỗi file đều có **23 cột** giống nhau → có thể gộp trực tiếp bằng `pd.concat`.

In [5]:
data_binh_thanh.head()

,tieu_de,gia_ban,don_gia,dien_tich,dia_chi,mo_ta,dien_thoai,loai_hinh,dien_tich_dat,dien_tich_su_dung,...,so_phong_ve_sinh,tong_so_tang,tinh_trang_noi_that,huong_cua_chinh,dac_diem,chieu_ngang,chieu_dai,ma_can,ten_phan_khu_lo,bieu_do_gia
0,🍀VÂN KIỀU AN CƯ🍀 40m2 - Hẻm Ô Tô - Gần mặt tiề...,"3,85 tỷ","106,94 triệu/m²",36 m²,"Đường Lê Quang Định, Phường 7, Quận Bình Thạnh...",🍀VÂN KIỀU AN CƯ🍀\r\n\r\n🏡 Nhà Lê Quang Định P5...,NaN,"Nhà ngõ, hẻm",36 m²,NaN,...,2 phòng,2.0,Nội thất đầy đủ,NaN,Nhà nở hậu,4.5 m,8 m,NaN,NaN,"[134.04, 125.8, 125.8, 125.8, 138.64, 138.64, ..."
1,"9.xty,Nhà Hẻm Xe Hơi,Xây Mới Khu VIP giáp Phạm...","9,79 tỷ","157,90 triệu/m²",62 m²,"Đường Phạm Văn Đồng, Phường 13, Quận Bình Thạn...","CHỈ 9.XTY - NHÀ MỚI KENG,XE HƠI NGỦ NHÀ GIÁP B...",0909761320,"Nhà ngõ, hẻm",62 m²,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[133.33, 135.54, 130.95, 130.02, 130.02, 136.3..."
2,XE HƠI NGỦ TRONG NHÀ- NGANG 7M - HIẾM - NƠ TRA...,"7,2 tỷ","133,33 triệu/m²",54 m²,"Đường Nơ Trang Long, Phường 13, Quận Bình Thạn...",XE HƠI NGỦ TRONG NHÀ- NGANG 7M - HIẾM - NƠ TRA...,0909983786,"Nhà ngõ, hẻm",54 m²,NaN,...,2 phòng,2.0,Hoàn thiện cơ bản,NaN,Hẻm xe hơi,7 m,7.7 m,NaN,NaN,"[168.37, 168.37, 120.71, 120.71, 124.24, 124.5..."
3,🛎️ NHÀ LỚN HẺM ÔTÔ NGAY HÀNG XANH – TIỆN XÂY M...,"8,5 tỷ","102,41 triệu/m²",83 m²,"Đường Xô Viết Nghệ Tĩnh, Phường 21, Quận Bình ...",#e45\r\n🛎️ NHÀ LỚN HẺM ÔTÔ NGAY HÀNG XANH – TI...,0969415765,"Nhà ngõ, hẻm",83 m²,NaN,...,NaN,NaN,NaN,NaN,NaN,7.5 m,11.5 m,NaN,NaN,"[159.22, 159.22, 157.45, 157.45, 135.08, 122.2..."
4,Nhà hẻm cách đường mặt tiền 20m,"2,85 tỷ","158,33 triệu/m²",18 m²,"Đường Chu Văn An, Phường 12, Quận Bình Thạnh, ...",Nhà hẻm cách đường mặt tiền 20m\r\nNhà ở khu v...,NaN,"Nhà ngõ, hẻm",18 m²,18 m²,...,1 phòng,NaN,NaN,NaN,Hiện trạng khác,3 m,6 m,NaN,NaN,"[132.67, 132.67, 169.64, 169.64, 133.33, 133.3..."


### Nhận xét
Mỗi bản ghi gồm 23 cột thông tin về bất động sản: tiêu đề, giá bán, đơn giá, diện tích, địa chỉ, mô tả, loại hình,...
Dữ liệu còn ở dạng text thô (giá bán = "3,85 tỷ", diện tích = "36 m²") → cần xử lý ở bước Data Cleaning.


In [6]:
data_binh_thanh.columns

Index(['tieu_de', 'gia_ban', 'don_gia', 'dien_tich', 'dia_chi', 'mo_ta',
       'dien_thoai', 'loai_hinh', 'dien_tich_dat', 'dien_tich_su_dung',
       'gia_m2', 'giay_to_phap_ly', 'so_phong_ngu', 'so_phong_ve_sinh',
       'tong_so_tang', 'tinh_trang_noi_that', 'huong_cua_chinh', 'dac_diem',
       'chieu_ngang', 'chieu_dai', 'ma_can', 'ten_phan_khu_lo', 'bieu_do_gia'],
      dtype='object')

In [7]:
data_binh_thanh["source"] = "quan-binh-thanh.csv"
data_go_vap["source"] = "quan-go-vap.csv"
data_phu_nhuan["source"] = "quan-phu-nhuan.csv"

In [8]:
def normalize_columns(df):
    """Chuẩn hóa tên cột: viết thường, bỏ khoảng trắng thừa, thay space bằng underscore."""
    df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
    return df

In [9]:
data_binh_thanh = normalize_columns(data_binh_thanh)
data_go_vap = normalize_columns(data_go_vap)
data_phu_nhuan = normalize_columns(data_phu_nhuan)

In [10]:
df = pd.concat([data_binh_thanh, data_go_vap, data_phu_nhuan], ignore_index=True)

In [11]:
df.shape

(8273, 24)

### Nhận xét

Sau khi gộp 3 file CSV, dataset có **8,273 dòng** và **24 cột** (23 cột gốc + 1 cột `source`).

Số dòng = 2,665 + 4,374 + 1,234 = 8,273 → khớp, không bị mất dữ liệu khi merge.

In [12]:
df.head()

,tieu_de,gia_ban,don_gia,dien_tich,dia_chi,mo_ta,dien_thoai,loai_hinh,dien_tich_dat,dien_tich_su_dung,...,tong_so_tang,tinh_trang_noi_that,huong_cua_chinh,dac_diem,chieu_ngang,chieu_dai,ma_can,ten_phan_khu_lo,bieu_do_gia,source
0,🍀VÂN KIỀU AN CƯ🍀 40m2 - Hẻm Ô Tô - Gần mặt tiề...,"3,85 tỷ","106,94 triệu/m²",36 m²,"Đường Lê Quang Định, Phường 7, Quận Bình Thạnh...",🍀VÂN KIỀU AN CƯ🍀\r\n\r\n🏡 Nhà Lê Quang Định P5...,NaN,"Nhà ngõ, hẻm",36 m²,NaN,...,2.0,Nội thất đầy đủ,NaN,Nhà nở hậu,4.5 m,8 m,NaN,NaN,"[134.04, 125.8, 125.8, 125.8, 138.64, 138.64, ...",quan-binh-thanh.csv
1,"9.xty,Nhà Hẻm Xe Hơi,Xây Mới Khu VIP giáp Phạm...","9,79 tỷ","157,90 triệu/m²",62 m²,"Đường Phạm Văn Đồng, Phường 13, Quận Bình Thạn...","CHỈ 9.XTY - NHÀ MỚI KENG,XE HƠI NGỦ NHÀ GIÁP B...",0909761320,"Nhà ngõ, hẻm",62 m²,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[133.33, 135.54, 130.95, 130.02, 130.02, 136.3...",quan-binh-thanh.csv
2,XE HƠI NGỦ TRONG NHÀ- NGANG 7M - HIẾM - NƠ TRA...,"7,2 tỷ","133,33 triệu/m²",54 m²,"Đường Nơ Trang Long, Phường 13, Quận Bình Thạn...",XE HƠI NGỦ TRONG NHÀ- NGANG 7M - HIẾM - NƠ TRA...,0909983786,"Nhà ngõ, hẻm",54 m²,NaN,...,2.0,Hoàn thiện cơ bản,NaN,Hẻm xe hơi,7 m,7.7 m,NaN,NaN,"[168.37, 168.37, 120.71, 120.71, 124.24, 124.5...",quan-binh-thanh.csv
3,🛎️ NHÀ LỚN HẺM ÔTÔ NGAY HÀNG XANH – TIỆN XÂY M...,"8,5 tỷ","102,41 triệu/m²",83 m²,"Đường Xô Viết Nghệ Tĩnh, Phường 21, Quận Bình ...",#e45\r\n🛎️ NHÀ LỚN HẺM ÔTÔ NGAY HÀNG XANH – TI...,0969415765,"Nhà ngõ, hẻm",83 m²,NaN,...,NaN,NaN,NaN,NaN,7.5 m,11.5 m,NaN,NaN,"[159.22, 159.22, 157.45, 157.45, 135.08, 122.2...",quan-binh-thanh.csv
4,Nhà hẻm cách đường mặt tiền 20m,"2,85 tỷ","158,33 triệu/m²",18 m²,"Đường Chu Văn An, Phường 12, Quận Bình Thạnh, ...",Nhà hẻm cách đường mặt tiền 20m\r\nNhà ở khu v...,NaN,"Nhà ngõ, hẻm",18 m²,18 m²,...,NaN,NaN,NaN,Hiện trạng khác,3 m,6 m,NaN,NaN,"[132.67, 132.67, 169.64, 169.64, 133.33, 133.3...",quan-binh-thanh.csv


In [13]:
df.to_csv(OUTPUT_PATH + "dataset_merged.csv", index=False, encoding="utf-8-sig")

In [14]:
import os
os.listdir(OUTPUT_PATH)

['dataset_cleaned.csv',
 'dataset_cleaned.parquet',
 'dataset_merged.csv',
 'data_modeling.csv',
 'data_modeling.parquet',
 'pyspark_predictions.parquet']

In [15]:
test_df = pd.read_csv(OUTPUT_PATH + "dataset_merged.csv")

In [16]:
test_df.shape

(8273, 24)

## Kết luận

### Kết quả đạt được
- Đã load thành công dữ liệu từ **3 file CSV** (Bình Thạnh, Gò Vấp, Phú Nhuận)
- Thêm cột `source` để truy vết nguồn gốc từng bản ghi
- Chuẩn hóa tên cột (lowercase, underscore)
- Gộp thành **1 dataset thống nhất**: `dataset_merged.csv`

### Thống kê dataset sau merge

| Nguồn | Số dòng | Tỷ lệ |
|-------|---------|--------|
| Gò Vấp | 4,374 | 52.9% |
| Bình Thạnh | 2,665 | 32.2% |
| Phú Nhuận | 1,234 | 14.9% |
| **Tổng** | **8,273** | **100%** |

- **Số cột**: 24 (23 cột gốc + 1 cột `source`)
- **File output**: `data/processed/dataset_merged.csv`

### Lưu ý
- Dataset chưa qua bất kỳ bước cleaning nào → sẽ được xử lý ở notebook `02_data_cleaning`
- Gò Vấp chiếm hơn 50% dataset → cần lưu ý khi phân tích để tránh bias
- Dữ liệu vẫn còn ở dạng text thô (giá, diện tích,...) → cần parse sang numeric